# **Day 48: Object Oriented Programming (Part 8) - Magic Methods in Depth**

On Day 42 we met `__init__`, `__str__` and `__repr__`. Today we see many more magic methods. They let **our own objects** work with Python's normal syntax like `len()`, `==`, `<`, `in` and `[]`.

| We write | Python actually calls |
|---|---|
| `len(x)` | `x.__len__()` |
| `x[i]` | `x.__getitem__(i)` |
| `print(x)` | `x.__str__()` |
| `x == y` | `x.__eq__(y)` |
| `x < y` | `x.__lt__(y)` |
| `a in x` | `x.__contains__(a)` |
| `x()` | `x.__call__()` |
| `bool(x)` | `x.__bool__()` |
| `hash(x)` | `x.__hash__()` |

In [1]:
# Even numbers and strings use magic methods inside
print((5).__add__(3))
print("kol".__len__())
print([1, 2, 3].__contains__(2))

8
3
True


## **Example: A Playlist Class**

In [2]:
class Playlist:
    def __init__(self, name, songs=None):
        self.name = name
        self.songs = list(songs or [])

    def __str__(self):
        return f"{self.name}:\n" + "\n".join(f"  {i}. {s}" for i, s in enumerate(self.songs, 1))

    def __repr__(self):
        return f"Playlist({self.name!r}, {len(self)} songs)"

    def __len__(self):
        return len(self.songs)

    def __getitem__(self, i):
        return self.songs[i]

    def __contains__(self, song):
        return song.lower() in (s.lower() for s in self.songs)

    def __bool__(self):
        return len(self.songs) > 0

    def __add__(self, other):
        return Playlist(f"{self.name} + {other.name}", self.songs + other.songs)

rabindra = Playlist("Rabindrasangeet", ["Ekla Cholo Re", "Purano Sei Diner Kotha", "Amaro Porano Jaha Chay"])
folk = Playlist("Bengali Folk", ["Bhawaiya", "Bhatiyali", "Baul"])

print(rabindra)
print(repr(folk))
print(len(rabindra), rabindra[0])
print("ekla cholo re" in rabindra)
print(bool(Playlist("Empty")))
print(repr(rabindra + folk))

Rabindrasangeet:
  1. Ekla Cholo Re
  2. Purano Sei Diner Kotha
  3. Amaro Porano Jaha Chay
Playlist('Bengali Folk', 3 songs)
3 Ekla Cholo Re
True
False
Playlist('Rabindrasangeet + Bengali Folk', 6 songs)


## **Example from GIS: A BoundingBox Class**
A **bounding box** is the rectangle that covers an area: `(min_lon, min_lat, max_lon, max_lat)`. GIS software uses bounding boxes all the time, for example to quickly check if a point might be inside an area.

Let us give our BoundingBox class some magic methods:
- `in` checks if a point is inside the box (`__contains__`).
- `&` finds the **overlap** of two boxes (`__and__`).
- `==` compares two boxes (`__eq__`).
- `bool()` is False for an empty box (`__bool__`).

In [3]:
class BoundingBox:
    def __init__(self, min_lon, min_lat, max_lon, max_lat):
        self.min_lon, self.min_lat = min_lon, min_lat
        self.max_lon, self.max_lat = max_lon, max_lat

    def __repr__(self):
        return f"BBox(lon {self.min_lon}..{self.max_lon}, lat {self.min_lat}..{self.max_lat})"

    def __contains__(self, point):
        lat, lon = point
        return self.min_lat <= lat <= self.max_lat and self.min_lon <= lon <= self.max_lon

    def __and__(self, other):
        """Overlap of two boxes (may be empty)."""
        return BoundingBox(max(self.min_lon, other.min_lon), max(self.min_lat, other.min_lat),
                           min(self.max_lon, other.max_lon), min(self.max_lat, other.max_lat))

    def __bool__(self):
        return self.min_lon < self.max_lon and self.min_lat < self.max_lat

    def __eq__(self, other):
        return (self.min_lon, self.min_lat, self.max_lon, self.max_lat) == \
               (other.min_lon, other.min_lat, other.max_lon, other.max_lat)

    def area_deg(self):
        return (self.max_lon - self.min_lon) * (self.max_lat - self.min_lat) if self else 0

In [4]:
west_bengal = BoundingBox(85.8, 21.5, 89.9, 27.3)
satellite_scene = BoundingBox(87.5, 21.0, 89.0, 22.8)     # Area covered by one image
bihar = BoundingBox(83.3, 24.3, 88.3, 27.5)
odisha_south = BoundingBox(81.0, 17.0, 83.0, 19.0)

print((22.57, 88.36) in west_bengal)      # Kolkata
print((19.81, 85.83) in west_bengal)      # Puri

overlap = west_bengal & satellite_scene
print("Overlap of WB and scene:", overlap)
print("Overlap area (square degrees):", round(overlap.area_deg(), 2))

print("WB and South Odisha overlap?", bool(west_bengal & odisha_south))
print("WB and Bihar overlap?", bool(west_bengal & bihar))

True
False
Overlap of WB and scene: BBox(lon 87.5..89.0, lat 21.5..22.8)
Overlap area (square degrees): 1.95
WB and South Odisha overlap? False
WB and Bihar overlap? True


Let us understand the code above:
- `(22.57, 88.36) in west_bengal` calls `__contains__`, which checks both latitude and longitude.
- `west_bengal & satellite_scene` calls `__and__`, which takes the bigger of the two minimums and the smaller of the two maximums.
- If the overlap box has no size, `__bool__` returns False. That is how we know two areas do not overlap.

## **__call__: Making an Object Work Like a Function**

In [5]:
class FareCalculator:
    def __init__(self, base, per_km):
        self.base = base
        self.per_km = per_km

    def __call__(self, km):
        return self.base + self.per_km * km

taxi = FareCalculator(30, 20)
toto = FareCalculator(10, 8)
print(taxi(5), toto(5))        # The objects are called like functions

130 50


## **Comparison Methods and total_ordering**
If we write `__eq__` and `__lt__`, the `@total_ordering` decorator creates `<=`, `>`, `>=` for us.

In [6]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, text):
        self.parts = tuple(int(x) for x in text.split("."))

    def __eq__(self, other):
        return self.parts == other.parts

    def __lt__(self, other):
        return self.parts < other.parts

    def __repr__(self):
        return ".".join(map(str, self.parts))

versions = [Version("3.10.2"), Version("3.9.18"), Version("3.12.0"), Version("3.10.12")]
print(sorted(versions))
print(Version("3.10.0") >= Version("3.9.9"))

[3.9.18, 3.10.2, 3.10.12, 3.12.0]
True


If we compared these as plain text, "3.10" would come before "3.9", which is wrong. Comparing them as tuples of numbers gives the correct order.

## **__hash__: Needed for Sets and Dictionary Keys**

In [7]:
class Station:
    def __init__(self, code, name):
        self.code, self.name = code, name
    def __eq__(self, other):
        return self.code == other.code
    def __hash__(self):
        return hash(self.code)
    def __repr__(self):
        return self.code

stations = {Station("HWH", "Howrah"), Station("SDAH", "Sealdah"), Station("HWH", "Howrah Jn")}
print(len(stations))       # Both HWH objects are treated as the same station

2


In [8]:
# Quick question
# Which magic method does each line call?
# len(obj), obj[0], obj == other, "x" in obj, print(obj)